# 04b Preprocessing experiments
Additional check, not required by the project: it tests the decisions of `02_preprocessing.ipynb` and the input format by training, with the chosen settings of `03_model1_embeddings.ipynb` and `04_model2_rnn.ipynb`.

**In short.** Question: does another preprocessing decision or input format clearly lower the validation loss? Answer: no preprocessing variant meets the decision rule, so the preprocessing stays; `per-choice` meets it for model 1. Decision: the selected models stay, and `05_evaluation.ipynb` evaluates `per-choice` for model 1 as a separate comparison.

`02_preprocessing.ipynb` justifies the preprocessing decisions with measurements such as the GloVe coverage; here they are tested by training. Each variant changes one decision of the baseline, and everything else stays the same: the chosen configuration of each model, seed 42, and the validation set for the comparison. The test set is not loaded.

| Decision | Baseline | Variants |
|---|---|---|
| Tokenization | `word_tokenize` | `whitespace-tokens` |
| Lowercasing | lowercase | `cased` |
| Stemming, lemmatization | none | `stemming`, `lemmatization` (with part-of-speech tags) |
| Stopwords | kept | `no-stopwords` |
| Punctuation | kept | `no-punctuation` |
| Unknown words | skipped | `unk-token`: one fixed vector, the average of the vocabulary |
| Rare words | kept | `no-rare-words`: fewer than 5 occurrences in the training data |
| Vocabulary | model 1: any GloVe word; model 2: training words | model 1: `train-vocab`; model 2: `glove-vocab` |
| Format cleaning | none | `format-cleaning`: curly and broken apostrophes become `'` |
| Truncation | model 1: none; model 2: 34 tokens | `truncate-20`, `truncate-10` |
| Feature selection | O1, O2, h1, h2 | `hypotheses-only`, `observations-only`; model 1 also `hypothesis-difference`: O1, O2, h1 - h2 |
| Input format | O1, O2, h1, h2 in fixed positions, 2 outputs | `per-choice`: the same classifier scores each hypothesis; model 2 also `one-pass`: one RNN pass per hypothesis over O1, O2, separator, hypothesis |

**Decision rule:** these runs check the decisions of `02_preprocessing.ipynb`; they are not a search for a new pipeline. A variant is looked at only if it has a lower validation loss and significantly more correct predictions than the baseline: only the variant right on more examples than only the baseline, with p < 0.05 in the exact McNemar test (as in `04_model2_rnn.ipynb`). Each variant has one run, so small differences can be chance, and with about 16 comparisons per model, one p below 0.05 can also occur by chance. The requirement of more correct predictions was added after the first model 1 runs, because the two-sided test alone could accept a significantly worse variant; it can only make the rule stricter. `hypotheses-only` and `observations-only` are diagnostics, not candidates, because the task is to choose the hypothesis that explains the observations. The validation set comes from the unfiltered original training data (see "Data" in `README.md`), so it can also reward decisions that exploit artifacts of the hypotheses.

**Code changed for a GPU run (1.11):** the GPU, patience 10 for every run, and the largest batch size that fits the GPU, up to 2048. The saved outputs, results and conclusions below are still those of the CPU run (commit `e5aaca6`) until this notebook is run again.

In [ ]:
# setup: on Colab, connect Google Drive; everywhere, use the GPU if there is one
import os
import sys

import torch

if "google.colab" in sys.modules:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/nlp-project1-art")
    %pip install -q gensim datasets wandb nltk
    # the key comes from Colab Secrets, so it is not in the notebook
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
# the W&B server of this project
os.environ["WANDB_BASE_URL"] = "https://api.forge.coreweave.com"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## Data, GloVe and resources
The training and validation splits (like on slide 9 of the course slides "Course Project HS26"), the GloVe vectors and the NLTK resources, as in `02_preprocessing.ipynb`. New is the part-of-speech tagger for the lemmatization.

In [1]:
import re
import string
import time

import gensim.downloader as api
import nltk
import numpy as np
from datasets import load_dataset

train = load_dataset("amarfurt/art-nlp-hs26", split="train")
valid = load_dataset("amarfurt/art-nlp-hs26", split="validation")
print(len(train), len(valid))

wv = api.load("glove-wiki-gigaword-300")

nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)


16801 1000


True

## Preprocessing variants
`preprocess()` is the function of `02_preprocessing.ipynb` with one change per variant. The tagger is loaded once, because `nltk.pos_tag` loads it again in every call. `is_known()` decides which words have a vector in a variant; all others are skipped, or replaced by the `<unk>` vector in the variant `unk-token`.

In [2]:
columns = ["observation_1", "observation_2", "hypothesis_1", "hypothesis_2"]

stopwords = set(nltk.corpus.stopwords.words("english"))
stemmer = nltk.stem.PorterStemmer()
lemmatizer = nltk.stem.WordNetLemmatizer()
tagger = nltk.tag.perceptron.PerceptronTagger()
# first letter of the part-of-speech tag -> part of speech of WordNet (default: noun)
wordnet_pos = {"J": "a", "V": "v", "R": "r"}

def is_punctuation(word):
    return all(character in string.punctuation for character in word)

# tokens of each text are kept, because most variants need the same ones
token_cache = {}

def tokenize(text):
    if text not in token_cache:
        token_cache[text] = nltk.word_tokenize(text)
    return token_cache[text]

# the baseline, with one difference per variant
def preprocess(text, variant):
    if variant == "format-cleaning":
        # curly and broken apostrophes become '
        text = re.sub("[\u2019\u00b4\ufffd]+", "'", text)
    words = text.split() if variant == "whitespace-tokens" else tokenize(text)
    if variant == "cased":
        return words
    words = [word.lower() for word in words]
    if variant == "stemming":
        return [stemmer.stem(word) for word in words]
    if variant == "lemmatization":
        return [lemmatizer.lemmatize(word, wordnet_pos.get(tag[0], "n")) for word, tag in tagger.tag(words)]
    if variant == "no-stopwords":
        return [word for word in words if word not in stopwords]
    if variant == "no-punctuation":
        return [word for word in words if not is_punctuation(word)]
    return words

# preprocessed words of all texts of a split, per column
def split_words(split, variant):
    return {column: [preprocess(text, variant) for text in split[column]] for column in columns}

# how often each word occurs in the training data of this variant
def word_counts(train_words):
    return nltk.FreqDist(word for column in columns for words in train_words[column] for word in words)

# does this word have a vector in this variant?
def is_known(word, variant, counts, model_number):
    if word not in wv:
        return False
    if variant == "no-rare-words":
        return counts[word] >= 5
    # vocabulary of the training data: baseline of model 2, variant of model 1
    if variant == "train-vocab" or (model_number == 2 and variant != "glove-vocab"):
        return word in counts
    return True

# the texts that a variant uses
def used_columns(variant):
    return {"hypotheses-only": columns[2:], "observations-only": columns[:2]}.get(variant, columns)

print(preprocess("He didn\u2019t go to the hospitals.", "baseline"))
print(preprocess("He didn\u2019t go to the hospitals.", "format-cleaning"))
print(preprocess("He didn't go to the hospitals.", "lemmatization"))


['he', 'didn', '’', 't', 'go', 'to', 'the', 'hospitals', '.']
['he', 'did', "n't", 'go', 'to', 'the', 'hospitals', '.']
['he', 'do', "n't", 'go', 'to', 'the', 'hospital', '.']


## Classes, evaluation and W&B
The classes, `evaluate()` and the loss as in `03_model1_embeddings.ipynb` and `04_model2_rnn.ipynb`. `RNNClassifier` has two changes that do not affect the baseline: `num_texts`, because the feature selection variants have 2 texts instead of 4, and `.clamp(min=1)`, because a variant can leave a text without any word, and `pack_padded_sequence` needs a length of at least 1. Such a text is one step of the zero vector.

In [3]:
import os

# no W&B log lines in the notebook output; the metrics are still logged
os.environ["WANDB_SILENT"] = "true"

import torch
import torch.nn as nn
import wandb
from scipy.stats import binomtest
from torch.nn.utils.rnn import pack_padded_sequence
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset

# same class as in 03_model1_embeddings.ipynb
class Classifier(nn.Module):

    def __init__(self, input_dim, output_dim, inner_dim=10):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, inner_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(inner_dim, output_dim)

    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        return out

# as in 04_model2_rnn.ipynb, with num_texts and clamp(min=1)
class RNNClassifier(nn.Module):

    def __init__(self, embedding_matrix, rnn_type="lstm", hidden_dim=128, num_layers=2, dropout=0.0, inner_dim=100, num_texts=4):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=True, padding_idx=0)
        embedding_dim = embedding_matrix.shape[1]
        if rnn_type == "lstm":
            self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        else:
            self.rnn = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = Classifier(num_texts * hidden_dim, 2, inner_dim=inner_dim)
        self.num_texts = num_texts

    def forward(self, x):
        features = []
        for i in range(self.num_texts):
            text = x[:, i]
            # real length of each text, at least 1
            lengths = (text != 0).sum(dim=1).clamp(min=1)
            embedded = self.embedding(text)
            packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
            _, hidden = self.rnn(packed)
            if isinstance(self.rnn, nn.LSTM):
                hidden = hidden[0]
            features.append(hidden[-1])
        features = torch.concat(features, dim=-1)
        features = self.dropout(features)
        return self.classifier(features)

criterion = nn.CrossEntropyLoss()

# the largest batch size that fits into the GPU memory, up to the limit, as in the course slides "Project Discussions"
# (Week 4), slide 14: the longest training example, one forward and backward pass, binary search
def batch_fits(build_model, X, y, index, size):
    try:
        model = build_model()
        x = X[index].repeat(size, *[1] * (X.dim() - 1))
        criterion(model(x), y[:1].repeat(size)).backward()
        del model, x
        torch.cuda.empty_cache()
        return True
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return False

# limit: a batch as large as the training data would make only one update per epoch
def largest_batch_size(build_model, X, y, limit=2048):
    # on a CPU, memory is not the limit: the batch size of the course exercise
    if device.type != "cuda":
        return 32
    # the example with the most real tokens
    index = (X != 0).reshape(len(X), -1).sum(dim=1).argmax()
    if batch_fits(build_model, X, y, index, limit):
        print("Largest batch size:", limit, "(the limit)")
        return limit
    # low fits, high does not; multiples of 32 leave a little memory free
    low, high = 32, limit
    while high - low > 32:
        middle = (low + high) // 2 // 32 * 32
        if batch_fits(build_model, X, y, index, middle):
            low = middle
        else:
            high = middle
    print("Largest batch size:", low)
    return low


# same function as in 03_model1_embeddings.ipynb
def evaluate(model, loader):
    model.eval()
    losses, predictions, labels = [], [], []
    with torch.no_grad():
        for x, y in loader:
            output = model(x)
            losses.append(criterion(output, y).item())
            predictions.append(output.argmax(dim=1))
            labels.append(y)
    predictions = torch.cat(predictions)
    labels = torch.cat(labels)
    accuracy = (predictions == labels).float().mean().item()
    return np.mean(losses), accuracy

# labels: 0 = h1, 1 = h2
y_train = torch.tensor([example - 1 for example in train["label"]]).to(device)
y_valid = torch.tensor([example - 1 for example in valid["label"]]).to(device)

wandb.login()

# one line per variant, sorted by the validation loss, with the exact McNemar test against the baseline
def print_results(results):
    correct_baseline = results["baseline"]["predictions"] == y_valid
    for variant, result in sorted(results.items(), key=lambda item: item[1]["loss"]):
        correct = result["predictions"] == y_valid
        # examples that only one of the two gets right
        only_variant = (correct & ~correct_baseline).sum().item()
        only_baseline = (~correct & correct_baseline).sum().item()
        p = binomtest(only_variant, only_variant + only_baseline, 0.5).pvalue if only_variant + only_baseline > 0 else 1.0
        print(f"{variant}: valid loss {result['loss']:.4f}, valid acc {result['accuracy']:.2%}, best epoch {result['epoch']}, only variant right {only_variant}, only baseline right {only_baseline}, p = {p:.4f}")


## Input format variants
The baseline gives h1 and h2 fixed input positions, so its prediction can depend on the position (`04_model2_rnn.ipynb`, section "Check: does the position of the hypotheses matter?"). Question: **does scoring each hypothesis separately lower the validation loss?** `per-choice` combines the pairs on slide 25 of the course slides "Project Discussions" (Week 5) with the one score per choice of the RNN example on slide 26; for averaged GloVe vectors, this is also the format of the bag-of-words baseline of Bhagavatula et al. (2020, Appendix A.4), which reports an accuracy of 50.5%. `one-pass` follows that RNN example. The output stays 2 logits, so the loss, the training and the evaluation do not change.
- `per-choice` (both models): the same 2-layer classifier scores [O1, O2, h1] and [O1, O2, h2], with 1 output each. In model 2, the RNN still reads each text separately.
- `one-pass` (model 2): the RNN reads [O1, O2, separator, h1] and [O1, O2, separator, h2], one pass each, so it reads each hypothesis after the observations. The same classifier scores each final hidden state.

Swapping h1 and h2 swaps the two logits by construction, which the check below shows.

In [4]:
# one score per hypothesis: the same classifier scores each hypothesis
class PerChoiceClassifier(nn.Module):

    def __init__(self, text_dim, inner_dim, num_observations=2):
        super().__init__()
        # size of the vector of one text (model 1: 300, model 2: RNN hidden size)
        self.text_dim = text_dim
        # how many observation vectors come before h1 and h2 (0 = none)
        self.num_observations = num_observations
        # input: the observations and one hypothesis; output: 1 score
        self.scorer = Classifier((num_observations + 1) * text_dim, 1, inner_dim=inner_dim)

    def forward(self, x):
        # x side by side: the observations, h1, h2
        start = self.num_observations * self.text_dim
        observations = x[:, :start]
        h1 = x[:, start:start + self.text_dim]
        h2 = x[:, start + self.text_dim:]
        # the same scorer for both hypotheses: [batch size, 1] each
        score1 = self.scorer(torch.concat([observations, h1], dim=1))
        score2 = self.scorer(torch.concat([observations, h2], dim=1))
        # the two scores are the 2 logits: [batch size, 2], as in the baseline
        return torch.concat([score1, score2], dim=1)


In [5]:
# untrained classifier on 5 made-up examples: 4 texts of 300 numbers side by side
torch.manual_seed(42)
test_model = PerChoiceClassifier(300, inner_dim=100)
x = torch.randn(5, 1200)
# the same examples with h1 and h2 swapped
swapped = torch.concat([x[:, :600], x[:, 900:], x[:, 600:900]], dim=1)
with torch.no_grad():
    logits = test_model(x)
    swapped_logits = test_model(swapped)
# 2 logits per example, as in the baseline
print(logits.shape)
# swapping the hypotheses must swap the logits
print(torch.equal(logits.flip(1), swapped_logits))
# number of parameters: per-choice (900 inputs, 1 output, used twice) and baseline (1200 inputs, 2 outputs)
print(sum(p.numel() for p in test_model.parameters()), sum(p.numel() for p in Classifier(1200, 2, inner_dim=100).parameters()))


torch.Size([5, 2])
True
90201 120302


## Model 1
The chosen configuration of `03_model1_embeddings.ipynb`: 100 hidden units, learning rate 0.000316, batch size 32, patience 10, as in its sweep. A W&B sweep (method `grid`) trains all variants automatically. The baseline must reproduce the features saved in `02_preprocessing.ipynb`, and the chosen run: same best epoch, validation loss and accuracy.

In [6]:
# one vector per text: the average of its word vectors, as in 02_preprocessing.ipynb
def model1_features(split_word_lists, variant, counts, unk_vector):
    max_length = {"truncate-20": 20, "truncate-10": 10}.get(variant)
    blocks = []
    for column in used_columns(variant):
        block = []
        for words in split_word_lists[column]:
            vectors = []
            for word in words:
                if is_known(word, variant, counts, 1):
                    vectors.append(wv[word])
                elif variant == "unk-token":
                    vectors.append(unk_vector)
            # [:None] keeps all words
            vectors = vectors[:max_length]
            block.append(np.mean(vectors, axis=0) if vectors else np.zeros(wv.vector_size))
        blocks.append(np.array(block, dtype=np.float32))
    if variant == "hypothesis-difference":
        # O1, O2, h1 - h2
        blocks = [blocks[0], blocks[1], blocks[2] - blocks[3]]
    return torch.tensor(np.concatenate(blocks, axis=1))

# all features of a variant
def prepare_model1(variant):
    train_words = split_words(train, variant)
    valid_words = split_words(valid, variant)
    counts = word_counts(train_words)
    # fixed vector for unknown words: the average of the vectors of the training words
    unk_vector = None
    if variant == "unk-token":
        unk_vector = np.mean([wv[word] for word in counts if word in wv], axis=0)
    return model1_features(train_words, variant, counts, unk_vector), model1_features(valid_words, variant, counts, unk_vector)

# check: the baseline must be the same as the saved features of model 1
X_train, X_valid = prepare_model1("baseline")
saved = np.load("features/model1.npz")
print("Baseline equals the saved features:", np.array_equal(X_train.numpy(), saved["X_train"]), np.array_equal(X_valid.numpy(), saved["X_valid"]))


Baseline equals the saved features: True True


In [7]:
variants_model1 = ["baseline", "whitespace-tokens", "cased", "stemming", "lemmatization", "no-stopwords", "no-punctuation",
                   "unk-token", "no-rare-words", "train-vocab", "format-cleaning", "truncate-20", "truncate-10",
                   "hypotheses-only", "observations-only", "hypothesis-difference", "per-choice"]

sweep_config_model1 = {
    "name": "model1-preprocessing",
    "method": "grid",
    "metric": {"name": "best_val_loss", "goal": "minimize"},
    "parameters": {"variant": {"values": variants_model1}},
}

# results of all variants, for the comparison below
results_model1 = {}

# one run: the W&B agent calls this function once per variant
MAX_BATCH1 = largest_batch_size(lambda: Classifier(1200, 2, inner_dim=100).to(device), X_train.to(device), y_train)

def run_model1(inner_dim=100, learning_rate=0.000316, batch_size=MAX_BATCH1, max_epochs=300, patience=10):
    run = wandb.init(project="nlp-project1-art-gpu")
    variant = run.config["variant"]
    name = f"prep-mlp-{variant}"
    run.name = name
    run.config.update({"model": "prep-mlp", "inner_dim": inner_dim, "learning_rate": learning_rate, "batch_size": batch_size, "max_epochs": max_epochs, "patience": patience, "seed": 42})
    checkpoint_path = f"checkpoints/{name}.pth"

    X_train, X_valid = prepare_model1(variant)
    X_train, X_valid = X_train.to(device), X_valid.to(device)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True, drop_last=True)
    train_eval_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=317)
    valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=100)

    # same training as in the sweep of 03_model1_embeddings.ipynb
    torch.manual_seed(42)
    if variant == "per-choice":
        # the same features, only the classifier differs
        model = PerChoiceClassifier(wv.vector_size, inner_dim=inner_dim).to(device)
    else:
        model = Classifier(X_train.shape[1], 2, inner_dim=inner_dim).to(device)
    optimizer = Adam(model.parameters(), lr=learning_rate)
    best_valid_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        model.train()
        for x, y in train_loader:
            output = model(x)
            loss = criterion(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        train_loss, train_acc = evaluate(model, train_eval_loader)
        valid_loss, valid_acc = evaluate(model, valid_loader)
        wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_valid_acc = valid_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement == patience:
                break

    wandb.run.summary["best_val_loss"] = best_valid_loss
    wandb.run.summary["best_val_acc"] = best_valid_acc
    wandb.run.summary["best_epoch"] = best_epoch
    wandb.finish()
    # predictions of the best epoch, for the comparison with the baseline
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    model.eval()
    with torch.no_grad():
        predictions = model(X_valid).argmax(dim=1)
    results_model1[variant] = {"loss": best_valid_loss, "accuracy": best_valid_acc, "epoch": best_epoch, "predictions": predictions}
    print(f"{name}: best epoch {best_epoch} of {epoch + 1}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")


In [8]:
# register the sweep at W&B and train all variants (a grid ends by itself)
sweep_id = wandb.sweep(sweep_config_model1, project="nlp-project1-art-gpu")
wandb.agent(sweep_id, function=run_model1)


Create sweep with ID: 8udpwoyf
Sweep URL: https://forge.coreweave.com/wandb/maria-micioni-hochschule-luzern/nlp-project1-art-final/sweeps/8udpwoyf
prep-mlp-baseline: best epoch 13 of 23, valid loss 0.6445, valid acc 63.20%
prep-mlp-whitespace-tokens: best epoch 6 of 16, valid loss 0.6637, valid acc 57.60%
prep-mlp-cased: best epoch 13 of 23, valid loss 0.6475, valid acc 61.60%
prep-mlp-stemming: best epoch 6 of 16, valid loss 0.6625, valid acc 58.20%
prep-mlp-lemmatization: best epoch 10 of 20, valid loss 0.6534, valid acc 59.40%
prep-mlp-no-stopwords: best epoch 6 of 16, valid loss 0.6408, valid acc 62.20%
prep-mlp-no-punctuation: best epoch 6 of 16, valid loss 0.6522, valid acc 60.90%
prep-mlp-unk-token: best epoch 8 of 18, valid loss 0.6484, valid acc 63.30%
prep-mlp-no-rare-words: best epoch 10 of 20, valid loss 0.6533, valid acc 62.00%
prep-mlp-train-vocab: best epoch 13 of 23, valid loss 0.6475, valid acc 63.00%
prep-mlp-format-cleaning: best epoch 13 of 23, valid loss 0.6473, va

In [9]:
print_results(results_model1)


per-choice: valid loss 0.5829, valid acc 69.10%, best epoch 32, only variant right 167, only baseline right 109, p = 0.0006
hypothesis-difference: valid loss 0.6173, valid acc 65.30%, best epoch 12, only variant right 116, only baseline right 95, p = 0.1684
hypotheses-only: valid loss 0.6347, valid acc 64.40%, best epoch 12, only variant right 100, only baseline right 88, p = 0.4225
no-stopwords: valid loss 0.6408, valid acc 62.20%, best epoch 6, only variant right 103, only baseline right 113, p = 0.5404
baseline: valid loss 0.6445, valid acc 63.20%, best epoch 13, only variant right 0, only baseline right 0, p = 1.0000
format-cleaning: valid loss 0.6473, valid acc 62.30%, best epoch 13, only variant right 47, only baseline right 56, p = 0.4307
cased: valid loss 0.6475, valid acc 61.60%, best epoch 13, only variant right 90, only baseline right 106, p = 0.2839
train-vocab: valid loss 0.6475, valid acc 63.00%, best epoch 13, only variant right 15, only baseline right 17, p = 0.8601
tru

### Result for model 1
The baseline reproduces the chosen run (best epoch 13, validation loss 0.6445, accuracy 63.20%). **No preprocessing variant meets the decision rule, so the preprocessing stays. One input format variant does: `per-choice`.** The p-values are from the exact McNemar test on the correct predictions, uncorrected unless stated.
- **Meets the rule:** `per-choice` (0.5829, 69.10%, best epoch 32): only the variant right on 167 examples, only the baseline on 109, p = 0.0006, also significant after the Holm correction. It has fewer trainable parameters than the baseline (90,201 vs. 120,302). The counts come from one pass over all examples and the accuracy from batches of 100. On 3 examples, h1 and h2 have nearly identical vectors, so their scores are tied up to rounding, and the two passes differ by one correct example (69.00% and 69.10%).
- **Worse than the baseline:** `whitespace-tokens` (p = 0.0004), `stemming` (p = 0.0043) and `lemmatization` (p = 0.0113): higher validation loss, lower accuracy. After a Holm correction over the 16 comparisons (Holm, 1979), only `whitespace-tokens` remains significant among these three.
- **No clear difference:** the other preprocessing variants have a higher validation loss, with p ≥ 0.09.
- **Lower validation loss, no significant accuracy gain:** `no-stopwords` (0.6408, 62.20%, p = 0.5404) and `hypothesis-difference` (0.6173, 65.30%, p = 0.1684), the second largest drop in validation loss (0.0272, after `per-choice` with 0.0616): a result for the discussion, not a change.
- **Sensitivity:** `format-cleaning` changes 36 of 67,204 training texts and no validation text, yet the validation loss moves by 0.0028 and 103 validation predictions differ. Differences of this size cannot be read as effects of a decision.
- **Feature selection:** `observations-only` stays at chance (50.60%); `hypotheses-only` shows no clear accuracy difference from all four texts (0.6347, 64.40%, p = 0.4225), as in the hypothesis-only run of `03_model1_embeddings.ipynb`.

## Model 2
The chosen configuration of `04_model2_rnn.ipynb`: GRU, RNN hidden size 128, learning rate 0.001, dropout 0.5, patience 3, at most 30 epochs. The vocabulary, the embedding matrix and the padding are built per variant as in `02_preprocessing.ipynb`. The baseline must reproduce the saved input of model 2, and the chosen run.

In [10]:
# token indices of all texts of a split: shape [examples, number of texts, max_length]
def model2_indices(split_word_lists, variant, vocab, unk_index, max_length):
    rows = []
    for column in used_columns(variant):
        block = []
        for words in split_word_lists[column]:
            numbers = []
            for word in words:
                if word in vocab:
                    numbers.append(vocab[word])
                elif variant == "unk-token":
                    numbers.append(unk_index)
            numbers = numbers[:max_length]
            block.append(numbers + [0] * (max_length - len(numbers)))
        rows.append(block)
    return torch.tensor(np.array(rows, dtype=np.int64).transpose(1, 0, 2))

# input and embedding matrix of a variant
def prepare_model2(variant):
    train_words = split_words(train, variant)
    valid_words = split_words(valid, variant)
    counts = word_counts(train_words)
    # all candidate words, the most frequent training words first, as in 02_preprocessing.ipynb
    candidates = [word for word, count in counts.most_common()]
    if variant == "glove-vocab":
        # also the validation words: the embeddings are pretrained, so this uses no labels
        candidates += [word for word in word_counts(valid_words) if word not in counts]
    vocab = {}
    for word in candidates:
        if is_known(word, variant, counts, 2):
            vocab[word] = len(vocab) + 1
    # one more index for <unk> in the variant unk-token, and for the separator in the variant one-pass
    unk_index = len(vocab) + 1
    size = len(vocab) + 2 if variant in ["unk-token", "one-pass"] else len(vocab) + 1
    matrix = np.zeros((size, wv.vector_size), dtype=np.float32)
    for word, number in vocab.items():
        matrix[number] = wv[word]
    if variant == "unk-token":
        # fixed vector for unknown words: the average of the vocabulary
        matrix[unk_index] = matrix[1:unk_index].mean(axis=0)
    if variant == "one-pass":
        # the frozen separator: one fixed random vector with the spread of the GloVe vectors
        matrix[unk_index] = np.random.default_rng(42).normal(0, matrix[1:unk_index].std(), wv.vector_size)
        # one sequence per hypothesis: O1, O2, separator, hypothesis; 62 = longest training sequence
        X_train = one_pass_indices(train_words, vocab, unk_index, 62)
        X_valid = one_pass_indices(valid_words, vocab, unk_index, 62)
        return X_train, X_valid, torch.tensor(matrix)
    max_length = {"truncate-20": 20, "truncate-10": 10}.get(variant, 34)
    X_train = model2_indices(train_words, variant, vocab, unk_index, max_length)
    X_valid = model2_indices(valid_words, variant, vocab, unk_index, max_length)
    return X_train, X_valid, torch.tensor(matrix)

# check: the baseline must be the same as the saved input of model 2
X_train, X_valid, embedding_matrix = prepare_model2("baseline")
saved = np.load("features/model2.npz")
print("Baseline equals the saved input:", np.array_equal(X_train.numpy(), saved["X_train"]), np.array_equal(X_valid.numpy(), saved["X_valid"]), np.array_equal(embedding_matrix.numpy(), saved["embedding_matrix"]))


Baseline equals the saved input: True True True


### Input of the variant `one-pass`
One sequence per hypothesis: the token indices of O1 and O2, the separator and the hypothesis, as in the format on slide 26 of the course slides "Project Discussions" (Week 5). The embeddings are frozen, so the separator is one fixed vector: random, with the same spread as the GloVe vectors. The padding length is 62 tokens, the longest such training sequence, and the padding is at the end, as for the other variants.

In [11]:
# token indices with one sequence per hypothesis: shape [examples, 2, max_length]
def one_pass_indices(split_word_lists, vocab, separator, max_length):
    rows = []
    # the 4 texts of one example at a time
    for o1, o2, h1, h2 in zip(*[split_word_lists[column] for column in columns]):
        # O1 and O2: only words in the vocabulary, as in model2_indices
        observations = [vocab[word] for word in o1 + o2 if word in vocab]
        sequences = []
        for hypothesis in [h1, h2]:
            # O1, O2, separator, hypothesis
            numbers = observations + [separator] + [vocab[word] for word in hypothesis if word in vocab]
            # cut and pad at the end, as in model2_indices
            numbers = numbers[:max_length]
            sequences.append(numbers + [0] * (max_length - len(numbers)))
        rows.append(sequences)
    return torch.tensor(np.array(rows, dtype=np.int64))


In [12]:
X_train_one, X_valid_one, matrix_one = prepare_model2("one-pass")
# shape [examples, 2, 62]; the matrix has one row more than the baseline: the separator
print(X_train_one.shape, X_valid_one.shape, matrix_one.shape)
# longest real sequence per split: no validation sequence may be longer than 62, or it would be cut
print((X_train_one != 0).sum(dim=-1).max().item(), (X_valid_one != 0).sum(dim=-1).max().item())
# the separator is the last row of the matrix; every sequence must contain it exactly once
separator = matrix_one.shape[0] - 1
print((X_train_one == separator).sum().item() == 2 * len(train), (X_valid_one == separator).sum().item() == 2 * len(valid))


torch.Size([16801, 2, 62]) torch.Size([1000, 2, 62]) torch.Size([15466, 300])
62 57
True True


In [13]:
variants_model2 = ["baseline", "whitespace-tokens", "cased", "stemming", "lemmatization", "no-stopwords", "no-punctuation",
                   "unk-token", "no-rare-words", "glove-vocab", "format-cleaning", "truncate-20", "truncate-10",
                   "hypotheses-only", "observations-only", "per-choice", "one-pass"]

sweep_config_model2 = {
    "name": "model2-preprocessing",
    "method": "grid",
    "metric": {"name": "best_val_loss", "goal": "minimize"},
    "parameters": {"variant": {"values": variants_model2}},
}

# results of all variants, for the comparison below
results_model2 = {}

# one run: the W&B agent calls this function once per variant
MAX_BATCH2 = largest_batch_size(lambda: RNNClassifier(matrix_one, rnn_type="gru", hidden_dim=128, dropout=0.5, num_texts=2).to(device), X_train_one.to(device), y_train)

def run_model2(hidden_dim=128, learning_rate=0.001, dropout=0.5, max_epochs=100, patience=10):
    start = time.time()
    run = wandb.init(project="nlp-project1-art-gpu")
    variant = run.config["variant"]
    name = f"prep-gru-{variant}"
    run.name = name
    run.config.update({"model": "prep-gru", "rnn_type": "gru", "hidden_dim": hidden_dim, "num_layers": 2, "inner_dim": 100, "learning_rate": learning_rate, "dropout": dropout, "max_epochs": max_epochs, "patience": patience, "batch_size": MAX_BATCH2, "seed": 42})
    checkpoint_path = f"checkpoints/{name}.pth"

    X_train, X_valid, embedding_matrix = prepare_model2(variant)
    X_train, X_valid = X_train.to(device), X_valid.to(device)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=MAX_BATCH2, shuffle=True, drop_last=True)
    train_eval_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=317)
    valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=100)

    # same training as in the sweep of 04_model2_rnn.ipynb
    torch.manual_seed(42)
    model = RNNClassifier(embedding_matrix, rnn_type="gru", hidden_dim=hidden_dim, dropout=dropout, num_texts=X_train.shape[1]).to(device)
    if variant == "per-choice":
        # the same RNN, only the classifier on the final hidden states differs
        model.classifier = PerChoiceClassifier(hidden_dim, inner_dim=100).to(device)
    if variant == "one-pass":
        # 2 sequences, one per hypothesis: the observations are already in each final hidden state
        model.classifier = PerChoiceClassifier(hidden_dim, inner_dim=100, num_observations=0).to(device)
    optimizer = Adam(model.parameters(), lr=learning_rate)
    best_valid_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        model.train()
        for x, y in train_loader:
            output = model(x)
            loss = criterion(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        train_loss, train_acc = evaluate(model, train_eval_loader)
        valid_loss, valid_acc = evaluate(model, valid_loader)
        wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_valid_acc = valid_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement == patience:
                break

    wandb.run.summary["best_val_loss"] = best_valid_loss
    wandb.run.summary["best_val_acc"] = best_valid_acc
    wandb.run.summary["best_epoch"] = best_epoch
    wandb.finish()
    # predictions of the best epoch, for the comparison with the baseline
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    model.eval()
    with torch.no_grad():
        predictions = model(X_valid).argmax(dim=1)
    results_model2[variant] = {"loss": best_valid_loss, "accuracy": best_valid_acc, "epoch": best_epoch, "predictions": predictions}
    minutes = (time.time() - start) / 60
    print(f"{name}: best epoch {best_epoch} of {epoch + 1}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%} ({minutes:.1f} min)")


In [14]:
# register the sweep at W&B and train all variants (a grid ends by itself)
sweep_id = wandb.sweep(sweep_config_model2, project="nlp-project1-art-gpu")
wandb.agent(sweep_id, function=run_model2)


Create sweep with ID: b5ij9ei2
Sweep URL: https://forge.coreweave.com/wandb/maria-micioni-hochschule-luzern/nlp-project1-art-final/sweeps/b5ij9ei2
prep-gru-baseline: best epoch 4 of 7, valid loss 0.5872, valid acc 67.60% (19.7 min)
prep-gru-whitespace-tokens: best epoch 4 of 7, valid loss 0.6132, valid acc 66.10% (12.5 min)
prep-gru-cased: best epoch 4 of 7, valid loss 0.5962, valid acc 66.70% (19.8 min)
prep-gru-stemming: best epoch 3 of 6, valid loss 0.6152, valid acc 64.50% (15.1 min)
prep-gru-lemmatization: best epoch 2 of 5, valid loss 0.6031, valid acc 64.00% (8.7 min)
prep-gru-no-stopwords: best epoch 3 of 6, valid loss 0.6083, valid acc 64.70% (8.2 min)
prep-gru-no-punctuation: best epoch 4 of 7, valid loss 0.5924, valid acc 66.50% (9.3 min)
prep-gru-unk-token: best epoch 4 of 7, valid loss 0.5881, valid acc 67.10% (10.5 min)
prep-gru-no-rare-words: best epoch 3 of 6, valid loss 0.6016, valid acc 67.20% (12.4 min)
prep-gru-glove-vocab: best epoch 4 of 7, valid loss 0.5866, vali

In [15]:
print_results(results_model2)


glove-vocab: valid loss 0.5866, valid acc 67.50%, best epoch 4, only variant right 17, only baseline right 18, p = 1.0000
baseline: valid loss 0.5872, valid acc 67.60%, best epoch 4, only variant right 0, only baseline right 0, p = 1.0000
unk-token: valid loss 0.5881, valid acc 67.10%, best epoch 4, only variant right 89, only baseline right 94, p = 0.7676
no-punctuation: valid loss 0.5924, valid acc 66.50%, best epoch 4, only variant right 90, only baseline right 101, p = 0.4694
one-pass: valid loss 0.5935, valid acc 68.70%, best epoch 4, only variant right 108, only baseline right 97, p = 0.4850
cased: valid loss 0.5962, valid acc 66.70%, best epoch 4, only variant right 94, only baseline right 103, p = 0.5688
truncate-10: valid loss 0.5977, valid acc 66.40%, best epoch 4, only variant right 86, only baseline right 98, p = 0.4175
truncate-20: valid loss 0.5987, valid acc 66.00%, best epoch 4, only variant right 87, only baseline right 103, p = 0.2765
format-cleaning: valid loss 0.599

### Result for model 2
The baseline reproduces the chosen run (best epoch 4, validation loss 0.5872, accuracy 67.60%). **No variant meets the decision rule, so the preprocessing stays.** Same test as for model 1.
- **Lower validation loss, no accuracy gain:** only `glove-vocab` (0.5866, 67.50%, p = 1.0000); the two runs differ on 35 validation examples.
- **Worse than the baseline:** `lemmatization` (0.6031, 64.00%, p = 0.0304), not significant after the Holm correction over the 16 comparisons; `no-stopwords` (0.6083, 64.70%, p = 0.0699) and `stemming` (0.6152, 64.50%, p = 0.0662) are close to it.
- **No clear difference:** the other preprocessing variants have a higher validation loss, with p ≥ 0.17.
- **Input format:** `one-pass` has a higher accuracy but also a higher validation loss (0.5935, 68.70%, p = 0.4850); `per-choice` is worse on both (0.6032, 66.10%, p = 0.2926).
- **Sensitivity:** `format-cleaning` changes 36 training texts and no validation text, yet the validation loss moves by 0.0126 and 192 validation predictions differ, more than for `unk-token`, `no-punctuation` and both truncations. Differences of this size cannot be read as effects of a decision.
- **Feature selection:** `observations-only` stays at chance (49.20%); `hypotheses-only` has a higher validation loss (0.6014 vs. 0.5872) at the same accuracy (67.70% vs. 67.60%, p = 1.0000).

## All results
Each row is one run with seed 42 on the validation set, so small differences can be chance; the p-values are in the outputs above.

| Variant | Model 1: loss / accuracy | Model 2: loss / accuracy |
|---|---|---|
| Baseline (the decisions of `02_preprocessing.ipynb`) | 0.6445 / 63.20% | 0.5872 / 67.60% |
| Whitespace tokens | 0.6637 / 57.60% | 0.6132 / 66.10% |
| Cased | 0.6475 / 61.60% | 0.5962 / 66.70% |
| Stemming | 0.6625 / 58.20% | 0.6152 / 64.50% |
| Lemmatization with part-of-speech tags | 0.6534 / 59.40% | 0.6031 / 64.00% |
| No stopwords | 0.6408 / 62.20% | 0.6083 / 64.70% |
| No punctuation | 0.6522 / 60.90% | 0.5924 / 66.50% |
| UNK vector | 0.6484 / 63.30% | 0.5881 / 67.10% |
| No rare words (fewer than 5 occurrences) | 0.6533 / 62.00% | 0.6016 / 67.20% |
| Other vocabulary (model 1: training words; model 2: also validation words with a GloVe vector) | 0.6475 / 63.00% | 0.5866 / 67.50% |
| Format cleaning | 0.6473 / 62.30% | 0.5998 / 65.60% |
| Truncation to 20 tokens | 0.6478 / 63.40% | 0.5987 / 66.00% |
| Truncation to 10 tokens | 0.6517 / 62.00% | 0.5977 / 66.40% |
| Hypotheses only (diagnostic) | 0.6347 / 64.40% | 0.6014 / 67.70% |
| Observations only (diagnostic) | 0.6930 / 50.60% | 0.6937 / 49.20% |
| O1, O2, h1 - h2 | 0.6173 / 65.30% | not tested |
| Each hypothesis scored separately (`per-choice`) | 0.5829 / 69.10% | 0.6032 / 66.10% |
| O1, O2, separator and one hypothesis as one sequence (`one-pass`) | not tested | 0.5935 / 68.70% |

## Conclusion
- **No preprocessing variant meets the decision rule for either model,** so the preprocessing of `02_preprocessing.ipynb` stays. The runs check the decisions; they do not show that the baseline is the best possible preprocessing.
- **Input format:** `per-choice` is the only variant that meets the rule, and only for model 1 (0.5829 vs. 0.6445, 69.10% vs. 63.20%, p = 0.0006). For model 2, neither `per-choice` nor `one-pass` has a lower validation loss than the baseline. The selected models stay: these runs are additional experiments and do not replace them, and `05_evaluation.ipynb` evaluates `per-choice` for model 1 as a separate comparison.
- **Feature selection:** `observations-only` is at chance in both models; `hypotheses-only` shows no clear accuracy difference from all four texts in either. In the baseline format, the observations show no measurable accuracy benefit on the unfiltered validation set. `per-choice` (0.5829) and `hypothesis-difference` (0.6173) have the lowest validation losses of the model 1 variants: the form of the input may matter more than the other decisions tested here.
- **Limits:** one run per variant (seed 42), one decision changed at a time, all comparisons on the unfiltered validation set.

## Sources
Course materials: NLP HS26 by A. Marfurt, Lucerne University of Applied Sciences and Arts (HSLU), 2026. Full references in README.md.
- Preprocessing decisions: course slides "Project Discussions" (Week 2), slide 10 (including "map to UNK"), and the course document "Grading criteria HS26" (A. Marfurt); baseline, `preprocess()`, the vocabulary, the padding and the embedding matrix from `02_preprocessing.ipynb` (see the sources there).
- Dataset: course version of ART, `amarfurt/art-nlp-hs26` on Hugging Face (A. Marfurt), based on Bhagavatula et al. (2020), https://arxiv.org/abs/1908.05739; loading the splits from slide 9 of the course slides "Course Project HS26".
- Word embeddings: GloVe (Pennington et al., 2014), loaded with gensim's downloader (Řehůřek & Sojka, 2010) as in the course exercise "Word Embeddings" (Week 2).
- Stemming and lemmatization: `nltk.stem.PorterStemmer` and `nltk.stem.WordNetLemmatizer` from the course slides "NLP Pipeline" (Week 1), slides 15 and 16; `lemmatize(word, pos)` from the NLTK documentation, https://www.nltk.org/api/nltk.stem.wordnet.html; part-of-speech tagger `PerceptronTagger` from the NLTK documentation, https://www.nltk.org/api/nltk.tag.perceptron.html, which `nltk.pos_tag` uses, https://www.nltk.org/api/nltk.tag.html; first letters of the Penn Treebank tags (J, V, R), Penn Treebank tag set as listed by `nltk.help.upenn_tagset`, https://www.nltk.org/api/nltk.help.html; downloading the resources with `nltk.download`, NLTK documentation "Installing NLTK Data", https://www.nltk.org/data.html.
- Format cleaning: `re.sub`, Python documentation, https://docs.python.org/3/library/re.html; the affected characters from the format check in `02_preprocessing.ipynb`.
- Models, training and evaluation: `Classifier`, `RNNClassifier` and `evaluate()` from `03_model1_embeddings.ipynb` and `04_model2_rnn.ipynb` (see the sources there); `Tensor.clamp`, PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.clamp.html; `numpy.transpose`, https://numpy.org/doc/stable/reference/generated/numpy.transpose.html, `numpy.array_equal`, https://numpy.org/doc/stable/reference/generated/numpy.array_equal.html, and `dict.get`, https://docs.python.org/3/library/stdtypes.html#dict.get.
- Sweep: W&B sweeps from the course slides "Project Discussions" (Week 7), slide 57, and the W&B documentation on the sweep configuration, https://docs.coreweave.com/models/sweeps/define-sweep-configuration, its options, https://docs.coreweave.com/models/sweeps/sweep-config-keys, and the sweeps walkthrough, https://docs.coreweave.com/models/sweeps/walkthrough; grid search from the same slides, slide 44.
- Comparison with the baseline: exact McNemar test (McNemar, 1947) with `scipy.stats.binomtest` as in `04_model2_rnn.ipynb` (see the sources there); `sorted(..., key=...)` with a `lambda`, Python documentation "Sorting Techniques", https://docs.python.org/3/howto/sorting.html.
- Correction for multiple comparisons: Holm, S. (1979), *A simple sequentially rejective multiple test procedure*, Scandinavian Journal of Statistics 6(2), 65–70, as described in "Holm–Bonferroni method", Wikipedia, https://en.wikipedia.org/wiki/Holm%E2%80%93Bonferroni_method.
- Input format variants: pairs per choice (slide 25), and one RNN pass per choice with a separator and one score per choice (slide 26), from the previous semester's examples in the course slides "Project Discussions" (Week 5); for averaged GloVe vectors also the bag-of-words baseline of Bhagavatula et al. (2020), Appendix A.4, https://arxiv.org/abs/1908.05739; separators as special tokens from the same slides, (Week 4), slide 15, and (Week 5), slide 20; `torch.concat` as in the course exercise "Sequence-to-sequence RNN" (Week 4); column indexing as in the course exercise "PyTorch" (Week 1); `torch.flip`, https://docs.pytorch.org/docs/stable/generated/torch.flip.html, `torch.randn`, https://docs.pytorch.org/docs/stable/generated/torch.randn.html, and `torch.max`, https://docs.pytorch.org/docs/stable/generated/torch.max.html, from the PyTorch documentation; `numpy.random.default_rng` and `Generator.normal`, NumPy documentation, https://numpy.org/doc/stable/reference/random/generator.html; `ndarray.std`, https://numpy.org/doc/stable/reference/generated/numpy.ndarray.std.html; `zip()` with `*`, Python documentation, https://docs.python.org/3/library/functions.html#zip.
- GPU: `torch.device`, `torch.cuda.is_available` and `Tensor.to`, PyTorch documentation "CUDA semantics", https://docs.pytorch.org/docs/stable/notes/cuda.html; `torch.cuda.empty_cache` and `torch.cuda.OutOfMemoryError`, https://docs.pytorch.org/docs/stable/cuda.html; `Tensor.repeat`, https://docs.pytorch.org/docs/stable/generated/torch.Tensor.repeat.html. HSLU GPUHub and Google Colab from the course slides "Course Project HS26", slide 11. The largest batch size, found with the longest example, one forward and backward pass and binary search, from the course slides "Project Discussions" (Week 4), slide 14, and (Week 7), slide 51. Environment variables on Colab: Google Colab Secrets and `drive.mount`, https://colab.research.google.com/.
- Tools: Claude (Anthropic), ChatGPT (OpenAI) and Gemini (Google), see "Use of AI" in README.md.
- Output of W&B: `WANDB_SILENT` (no W&B log lines in the notebook output), W&B documentation "Environment variables", https://docs.coreweave.com/models/track/environment-variables; `os.environ`, Python documentation, https://docs.python.org/3/library/os.html#os.environ.